# CodePath AI — Salary Model Baseline

In [1]:
import numpy as np
import pandas as pd

In [2]:
df_raw = pd.read_csv("../../data/raw/stackoverflow_full.csv")

In [3]:
df_clean = (
    df_raw
    .drop(columns=["Unnamed: 0"])
    .drop_duplicates()
    .copy()
)

In [4]:
salary_eligible_mask = df_clean["PreviousSalary"] >= 1000

salary_eligible_df = df_clean.loc[
    salary_eligible_mask
].copy()

In [5]:
valid_experience_mask = (
    salary_eligible_df["YearsCodePro"]
    <= salary_eligible_df["YearsCode"]
)

experience_valid_df = salary_eligible_df.loc[
    valid_experience_mask
].copy()

In [6]:
salary_features = [
    "Country",
    "EdLevel",
    "YearsCode",
    "YearsCodePro",
    "ComputerSkills"
]

salary_target = "PreviousSalary"

salary_df = experience_valid_df[
    salary_features + [salary_target]
].copy()

salary_df.head()

,Country,EdLevel,YearsCode,YearsCodePro,ComputerSkills,PreviousSalary
0,Sweden,Master,7,4,4,51552.0
1,Spain,Undergraduate,12,5,12,46482.0
2,Germany,Master,15,6,7,77290.0
3,Canada,Undergraduate,9,6,13,46135.0
4,Singapore,PhD,40,30,2,160932.0


In [7]:
dataset_sizes = {
    "raw_rows": len(df_raw),
    "after_deduplication": len(df_clean),
    "salary_model_rows": len(salary_df)
}

dataset_sizes

{'raw_rows': 73462, 'after_deduplication': 73458, 'salary_model_rows': 72396}

## Train/Test Split

In [8]:
from sklearn.model_selection import train_test_split

In [9]:
x = salary_df.drop("PreviousSalary", axis=1)
y = salary_df.PreviousSalary

In [10]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [11]:
train_test_data_shape = {
    "x_train": x_train.shape,
    "x_test": x_test.shape,
    "y_train": y_train.shape,
    "y_test": y_test.shape,

}

train_test_data_shape

{'x_train': (57916, 5),
 'x_test': (14480, 5),
 'y_train': (57916,),
 'y_test': (14480,)}

## Preprocessing Pipeline

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [13]:
categorical_features = ["Country", "EdLevel"]
numerical_features = ["YearsCode", "YearsCodePro", "ComputerSkills"]

In [14]:
salary_preprocessor = ColumnTransformer(
    transformers = (
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features,
        )),
        remainder="drop"
    )

salary_preprocessor

ColumnTransformer(transformers=(('categorical',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['Country', 'EdLevel']),
                                ('numerical', StandardScaler(),
                                 ['YearsCode', 'YearsCodePro',
                                  'ComputerSkills'])))

## Baseline Model — DummyRegressor

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [16]:
dummy_salary_pipeline = Pipeline(
    steps= [
        ("preprocessor", salary_preprocessor),
        ("model", DummyRegressor(strategy="median"))
    ]
)

In [17]:
dummy_salary_pipeline.fit(x_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=(('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country', 'EdLevel']),
                                                 ('numerical', StandardScaler(),
                                                  ['YearsCode', 'YearsCodePro',
                                                   'ComputerSkills'])))),
                ('model', DummyRegressor(strategy='median'))])

In [18]:
dummy_predictions = dummy_salary_pipeline.predict(x_test)

dummy_metrics = {
    "mae": mean_absolute_error(y_test, dummy_predictions),
    "rmse": float(np.sqrt(mean_squared_error(y_test, dummy_predictions))),
    "r2_score": r2_score(y_test, dummy_predictions),
}

dummy_metrics

{'mae': 38660.20303867403,
 'rmse': 50261.33238136588,
 'r2_score': -0.04661046302168792}

## Linear Baseline — Ridge Regression

In [19]:
from sklearn.linear_model import Ridge

In [20]:
ridge_salary_pipeline = Pipeline(
    steps = [
        ("preprocessor", salary_preprocessor),
        ("model", Ridge(alpha=1.0))
    ]
)

In [21]:
ridge_salary_pipeline.fit(x_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=(('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country', 'EdLevel']),
                                                 ('numerical', StandardScaler(),
                                                  ['YearsCode', 'YearsCodePro',
                                                   'ComputerSkills'])))),
                ('model', Ridge())])

In [22]:
ridge_predictions = ridge_salary_pipeline.predict(x_test)

ridge_metrics = {
    "mae": mean_absolute_error(y_test, ridge_predictions),
    "rmse": float(np.sqrt(mean_squared_error(y_test, ridge_predictions))),
    "r2_score": r2_score(y_test, ridge_predictions),
}

ridge_metrics

{'mae': 22568.87638928891,
 'rmse': 31757.612308628402,
 'r2_score': 0.5821573540030025}

## Baseline Model Comparison

In [23]:
model_comparison = pd.DataFrame(
    [dummy_metrics, ridge_metrics],
    index=["DummyRegressor", "Ridge"]
).round({
    "mae": 2,
    "rmse": 2,
    "r2_score": 4
})

model_comparison

,mae,rmse,r2_score
DummyRegressor,38660.20,50261.33,-0.0466
Ridge,22568.88,31757.61,0.5822


In [24]:
mae_improvement_percentage = (
    (dummy_metrics["mae"] - ridge_metrics["mae"])
    / dummy_metrics["mae"]
    * 100
)

round(mae_improvement_percentage, 2)

41.62

## Nonlinear Model — HistGradientBoostingRegressor

In [25]:
from sklearn.ensemble import HistGradientBoostingRegressor

In [26]:
tree_salary_preprocessor = ColumnTransformer(
    transformers=(
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    )
)

In [27]:
hist_gradient_salary_pipeline = Pipeline(
    steps=(
        ("preprocessor", tree_salary_preprocessor),
        ("model", HistGradientBoostingRegressor(
            loss="absolute_error",
            max_iter=200,
            learning_rate=0.08,
            max_leaf_nodes=31,
            min_samples_leaf=20,
            l2_regularization=1.0,
            random_state=42
        ))
    )
)

In [28]:
hist_gradient_salary_pipeline.fit(x_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=(('categorical',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['Country', 'EdLevel']),
                                                 ('numerical', 'passthrough',
                                                  ['YearsCode', 'YearsCodePro',
                                                   'ComputerSkills'])))),
                ('model',
                 HistGradientBoostingRegressor(l2_regularization=1.0,
                                               learning_rate=0.08,
                                               loss='absolute_error',
                                               max_iter=200,
                                               random_state=42))])

In [29]:
hist_gradient_predictions = hist_gradient_salary_pipeline.predict(x_test)

In [30]:
hist_gradient_metrics = {
    "mae": mean_absolute_error(y_test, hist_gradient_predictions),
    "rmse": float(np.sqrt(mean_squared_error(y_test, hist_gradient_predictions))),
    "r2_score": r2_score(y_test, hist_gradient_predictions)
}

hist_gradient_metrics

{'mae': 21035.316706176094,
 'rmse': 31183.13275699511,
 'r2_score': 0.5971377582212344}

In [31]:
lower_salary_pipeline = Pipeline(steps=(
    ("preprocessor", tree_salary_preprocessor),
    ("model", HistGradientBoostingRegressor(
        loss="quantile",
        quantile=0.1,
        max_iter=200,
        learning_rate=0.08,
        max_leaf_nodes=31,
        min_samples_leaf=20,
        l2_regularization=1.0,
        random_state=42
    ))
))

upper_salary_pipeline = Pipeline(steps=(
    ("preprocessor", tree_salary_preprocessor),
    ("model", HistGradientBoostingRegressor(
        loss="quantile",
        quantile=0.9,
        max_iter=200,
        learning_rate=0.08,
        max_leaf_nodes=31,
        min_samples_leaf=20,
        l2_regularization=1.0,
        random_state=42
    ))
))

In [32]:
lower_salary_pipeline.fit(x_train, y_train)
upper_salary_pipeline.fit(x_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=(('categorical',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['Country', 'EdLevel']),
                                                 ('numerical', 'passthrough',
                                                  ['YearsCode', 'YearsCodePro',
                                                   'ComputerSkills'])))),
                ('model',
                 HistGradientBoostingRegressor(l2_regularization=1.0,
                                               learning_rate=0.08,
                                               loss='quantile', max_iter=200,
                                               quantile=0.9,
                                               random_state=42))])

In [33]:
lower_predictions = lower_salary_pipeline.predict(x_test)
upper_predictions = upper_salary_pipeline.predict(x_test)

In [34]:
actual_values = y_test.to_numpy()

interval_contains_actual = (
    (actual_values >= lower_predictions)
    & (actual_values <= upper_predictions)
)

In [35]:
interval_metrics = {
    "coverage_percentage": round(
        float(interval_contains_actual.mean() * 100),
        2
    ),
    "average_interval_width": round(
        float((upper_predictions - lower_predictions).mean()),
        2
    ),
    "invalid_interval_count": int(
        (lower_predictions > upper_predictions).sum()
    ),
}

interval_metrics

{'coverage_percentage': 78.9,
 'average_interval_width': 66369.5,
 'invalid_interval_count': 0}

## Final Training and Artifact Export

In [36]:
import joblib
from pathlib import Path

In [37]:
hist_gradient_salary_pipeline.fit(x, y)
lower_salary_pipeline.fit(x, y)
upper_salary_pipeline.fit(x, y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=(('categorical',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['Country', 'EdLevel']),
                                                 ('numerical', 'passthrough',
                                                  ['YearsCode', 'YearsCodePro',
                                                   'ComputerSkills'])))),
                ('model',
                 HistGradientBoostingRegressor(l2_regularization=1.0,
                                               learning_rate=0.08,
                                               loss='quantile', max_iter=200,
                                               quantile=0.9,
                                               random_state=42))])

In [38]:
salary_model_bundle = {
    "point_model": hist_gradient_salary_pipeline,
    "lower_model": lower_salary_pipeline,
    "upper_model": upper_salary_pipeline,
    "features": salary_features,
    "model_version": "1.0.0",
    "interval_quantiles": [0.10, 0.90],
    "training_rows": len(salary_df),
}

salary_model_bundle

{'point_model': Pipeline(steps=[('preprocessor',
                  ColumnTransformer(transformers=(('categorical',
                                                   OneHotEncoder(handle_unknown='ignore',
                                                                 sparse_output=False),
                                                   ['Country', 'EdLevel']),
                                                  ('numerical', 'passthrough',
                                                   ['YearsCode', 'YearsCodePro',
                                                    'ComputerSkills'])))),
                 ('model',
                  HistGradientBoostingRegressor(l2_regularization=1.0,
                                                learning_rate=0.08,
                                                loss='absolute_error',
                                                max_iter=200,
                                                random_state=42))]),
 'lower_model': Pipeline(step

In [39]:
artifact_directory = Path("../../artifacts/salary")

artifact_directory.mkdir(
    parents=True,
    exist_ok=True
)

In [40]:
artifact_path = (
    artifact_directory
    / "salary_model_bundle_v1.joblib"
)

In [41]:
joblib.dump(
    salary_model_bundle,
    artifact_path
)

artifact_check = {
    "absolute_path": str(artifact_path.resolve()),
    "file_exists": artifact_path.exists(),
    "file_size_bytes": artifact_path.stat().st_size
}

artifact_check

{'absolute_path': '/Users/ccakir/Desktop/codepath_ai/artifacts/salary/salary_model_bundle_v1.joblib',
 'file_exists': True,
 'file_size_bytes': 2048642}

In [42]:
loaded_salary_bundle = joblib.load(artifact_path)

loaded_salary_bundle.keys()

dict_keys(['point_model', 'lower_model', 'upper_model', 'features', 'model_version', 'interval_quantiles', 'training_rows'])

In [43]:
sample_profile_dict = {
    "Country": "Germany",
    "EdLevel": "Master",
    "YearsCode": 8,
    "YearsCodePro": 3,
    "ComputerSkills": 10,
}

sample_profile = pd.DataFrame(
    sample_profile_dict,
    index=[0]
)

sample_profile

,Country,EdLevel,YearsCode,YearsCodePro,ComputerSkills
0,Germany,Master,8,3,10


In [44]:
predicted_salary = float(
    loaded_salary_bundle["point_model"]
    .predict(sample_profile)[0]
)

lower_salary = float(
    loaded_salary_bundle["lower_model"]
    .predict(sample_profile)[0]
)

upper_salary = float(
    loaded_salary_bundle["upper_model"]
    .predict(sample_profile)[0]
)

In [45]:
sample_prediction = {
    "predicted_salary": round(predicted_salary, 2),
    "lower_salary": round(lower_salary, 2),
    "upper_salary": round(upper_salary, 2),
    "model_version": loaded_salary_bundle["model_version"],
    "valid_interval": (
        lower_salary <= predicted_salary <= upper_salary
    )
}

sample_prediction

{'predicted_salary': 60440.71,
 'lower_salary': 38212.03,
 'upper_salary': 78516.64,
 'model_version': '1.0.0',
 'valid_interval': True}